# 🏥 Baseline Exploration Notebook - Medical Document Retrieval (R2AI 2026)

Notebook này hỗ trợ các thành viên trong đội:
1. Khám phá và kiểm tra pipeline tiền xử lý & phân đoạn (Chunking)
2. Thử nghiệm trích xuất đặc trưng Dense (BGE-M3) và BM25 Sparse Search
3. Kiểm tra tính năng Reciprocal Rank Fusion (RRF) & Reranking
4. Tính toán metric Macro F2 theo đặc tả của cuộc thi

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.config import load_config
from src.evaluation.metrics import compute_prf
from src.ingestion.chunker import DocumentChunker
from src.ingestion.cleaner import normalize_text

config = load_config("../configs/config.yaml")
print("Loaded configuration:", config)

Loaded configuration: paths=PathsConfig(raw_data_dir=PosixPath('data/raw'), processed_data_dir=PosixPath('data/processed'), indices_dir=PosixPath('data/indices'), outputs_dir=PosixPath('outputs'), submissions_dir=PosixPath('outputs/submissions')) crawler=CrawlerConfig(user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36', timeout_seconds=15, max_retries=3, concurrency=10, ncbi_email='team@r2ai2026.org', ncbi_api_key=None) chunking=ChunkingConfig(max_chunk_size=512, chunk_overlap=64, min_chunk_size=50, split_by_sentences=True) embedding=EmbeddingConfig(model_name='BAAI/bge-m3', batch_size=16, max_length=512, normalize_embeddings=True, device='auto', use_fp16=True) retrieval=RetrievalConfig(engine='qdrant', qdrant_path=PosixPath('data/indices/qdrant_db'), collection_name='medical_chunks', dense_top_k=50, sparse_top_k=50, fusion_method='rrf', rrf_k=60, dense_weight=0.6, sparse_weight=0.4, hybrid_top_k=30) reranker=Rera

## 1. Thử nghiệm Chunking tài liệu đa ngôn ngữ

In [2]:
chunker = DocumentChunker(
    max_chunk_size=config.chunking.max_chunk_size,
    chunk_overlap=config.chunking.chunk_overlap,
    min_chunk_size=config.chunking.min_chunk_size,
)

sample_doc = """Bệnh sỏi thận (sỏi niệu) là sự hình thành các tinh thể khoáng chất trong thận.
Khi kích thước sỏi tăng lên, nó có thể di chuyển và gây tắc nghẽn đường dẫn nước tiểu (niệu quản),
dẫn tới cơn đau quặn thận dữ dội, tiểu ra máu hoặc nhiễm trùng đường tiết niệu.
Các phương pháp điều trị bao gồm: uống nhiều nước, dùng thuốc giãn cơ trơn,
tán sỏi ngoài cơ thể (ESWL) hoặc nội soi tán sỏi qua da."""

chunks = chunker.chunk_document(doc_id="sample_vi_1", text=sample_doc, lang="vi")
print(f"Total chunks generated: {len(chunks)}")
for c in chunks:
    print(f"- [{c.chunk_id}] (len={len(c.chunk_text)}): {c.chunk_text[:100]}...")

Total chunks generated: 1
- [sample_vi_1__c0] (len=390): Bệnh sỏi thận (sỏi niệu) là sự hình thành các tinh thể khoáng chất trong thận.
Khi kích thước sỏi tă...


In [3]:
sample_doc[:chunker._find_split_point(text=sample_doc, target_end=500)]

'Bệnh sỏi thận (sỏi niệu) là sự hình thành các tinh thể khoáng chất trong thận.\nKhi kích thước sỏi tăng lên, nó có thể di chuyển và gây tắc nghẽn đường dẫn nước tiểu (niệu quản),\ndẫn tới cơn đau quặn thận dữ dội, tiểu ra máu hoặc nhiễm trùng đường tiết niệu.\nCác phương pháp điều trị bao gồm: uống nhiều nước, dùng thuốc giãn cơ trơn,\ntán sỏi ngoài cơ thể (ESWL) hoặc nội soi tán sỏi qua da.'

## 2. Kiểm tra tính điểm Macro F2 (Beta = 2)

In [4]:
retrieved_docs = ["doc_1", "doc_2", "doc_3"]
ground_truth_docs = ["doc_1", "doc_2", "doc_4"]

p, r, f2 = compute_prf(set(retrieved_docs), set(ground_truth_docs), beta=2.0)
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
print(f"F2 Score:  {f2:.4f}")

Precision: 0.6667
Recall:    0.6667
F2 Score:  0.6667


## 3. Normalized text

In [5]:
from src.ingestion.cleaner import normalize_text, detect_language

In [6]:
text = "Bắt đầu từ đây để hiểu các tham số và luật chơi của hệ thống."
normalized_text = normalize_text(text)
normalized_text

'Bắt đầu từ đây để hiểu các tham số và luật chơi của hệ thống.'

In [7]:
detected_lang = detect_language(normalized_text)
detected_lang

'vi'

In [8]:
detect_language("输尿管结石引起的尿路梗阻治疗原则")

'zh'

In [9]:
from src.crawler.query_translator import QueryTranslator

query_translator = QueryTranslator()

In [10]:
query_translator._load_model()

/home/thienhb/Workspace/med-doc-retrieval/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-26 13:01:40.527 | INFO     | src.crawler.query_translator:_load_model:82 - Loading translation model weights from Helsinki-NLP/opus-mt-vi-en...
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 3116.34it/s]


In [11]:
sample_doc

'Bệnh sỏi thận (sỏi niệu) là sự hình thành các tinh thể khoáng chất trong thận.\nKhi kích thước sỏi tăng lên, nó có thể di chuyển và gây tắc nghẽn đường dẫn nước tiểu (niệu quản),\ndẫn tới cơn đau quặn thận dữ dội, tiểu ra máu hoặc nhiễm trùng đường tiết niệu.\nCác phương pháp điều trị bao gồm: uống nhiều nước, dùng thuốc giãn cơ trơn,\ntán sỏi ngoài cơ thể (ESWL) hoặc nội soi tán sỏi qua da.'

In [12]:
query_translator.translate_to_english(sample_doc)

'Kidney stones are the formation of minerals in the kidneys. When they increase in size, they can move and disrupt urethras, resulting in severe kidney pain, urine, or urinary infections. The treatment includes drinking many water, evaluating drugs, scattering rocks (esWL), or looking through the skin with marbles.'

In [13]:
query_translator.extract_pubmed_keywords(sample_doc)

'kidney stones nephrolithiasis renal colic urinary tract infection'

In [15]:
from src.crawler.url_scraper import ArticleScraper

url_scraper = ArticleScraper()

In [19]:
import httpx
import asyncio
from tqdm.asyncio import tqdm

In [21]:
results = []
async with httpx.AsyncClient(timeout=10) as client:
    text = await url_scraper.fetch_url(client, "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1234567/")


In [22]:
text

'\n<!DOCTYPE html>\n<html lang="en" >\n    <head >\n\n        <meta charset="UTF-8" />\n        <meta http-equiv="X-UA-Compatible" content="IE=edge" />\n        <meta name="HandheldFriendly" content="True" />\n        <meta name="MobileOptimized" content="320" />\n        <meta name="viewport" content="width=device-width, initial-scale=1.0" />\n\n        \n        \n\n        \n        \n  <link  rel="stylesheet" href="https://cdn.ncbi.nlm.nih.gov/pmc/pd-medc-pmc-cloudpmc-viewer/production/9b1d9d87/var/data/static/assets/base_style-BxRs1iOp.css" />\n<script type="module" crossorigin="" src="https://cdn.ncbi.nlm.nih.gov/pmc/pd-medc-pmc-cloudpmc-viewer/production/9b1d9d87/var/data/static/assets/base_style-D-jH62-1.js"></script>\n\n  <link  rel="stylesheet" href="https://cdn.ncbi.nlm.nih.gov/pmc/pd-medc-pmc-cloudpmc-viewer/production/9b1d9d87/var/data/static/assets/article_style-B67IZFds.css" />\n<link  rel="stylesheet" href="https://cdn.ncbi.nlm.nih.gov/pmc/pd-medc-pmc-cloudpmc-viewer/pr

In [24]:
extracted_text = url_scraper.extract_text(text)

In [26]:
extracted_text

{'title': 'Intravenous contrast medium aggravates the impairment of pancreatic microcirculation in necrotizing pancreatitis in the rat',
 'text': "Abstract\nBACKGROUND: Previous reports demonstrated that radiographic contrast medium, as used in contrast-enhanced computed tomography, increases acinar necrosis and mortality in experimental pancreatitis. The authors studied the possibility that these changes may be related to an additional impairment of pancreatic microcirculation. METHODS: Fifty Wistar rats had acute pancreatitis induced by intraductal glycodeoxycholic acid (10 mmol/L for 10 min) and intravenous cerulein (5 micrograms/kg/hr for 6 hrs). After rehydration (16 mL/kg), pancreatic capillary perfusion was quantified by means of intravital microscopy at baseline before intravenous infusion of contrast medium (n = 25) or saline (n = 25), and 30 and 60 minutes thereafter. In addition to total capillary flow, capillaries were categorized as high- or low-flow (> or < 1.6 nL/min). R

In [27]:
{
    "doc_id": 1,
    "url": "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1234567/",
    "title": extracted_text.get("title", ""),
    "text": extracted_text.get("text", ""),
    "status": "success" if extracted_text.get("text") else "empty",
}

{'doc_id': 1,
 'url': 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1234567/',
 'title': 'Intravenous contrast medium aggravates the impairment of pancreatic microcirculation in necrotizing pancreatitis in the rat',
 'text': "Abstract\nBACKGROUND: Previous reports demonstrated that radiographic contrast medium, as used in contrast-enhanced computed tomography, increases acinar necrosis and mortality in experimental pancreatitis. The authors studied the possibility that these changes may be related to an additional impairment of pancreatic microcirculation. METHODS: Fifty Wistar rats had acute pancreatitis induced by intraductal glycodeoxycholic acid (10 mmol/L for 10 min) and intravenous cerulein (5 micrograms/kg/hr for 6 hrs). After rehydration (16 mL/kg), pancreatic capillary perfusion was quantified by means of intravital microscopy at baseline before intravenous infusion of contrast medium (n = 25) or saline (n = 25), and 30 and 60 minutes thereafter. In addition to total capillary

In [29]:
from src.embedding.bge_m3 import BGEM3Embedder
bge_m3 = BGEM3Embedder()

2026-09-26 13:35:52.821 | INFO     | src.embedding.bge_m3:__init__:32 - Initializing BGEM3Embedder with model='BAAI/bge-m3' on device='cuda', fp16=True


In [30]:
bge_m3.model

2026-09-26 13:36:23.279 | INFO     | src.embedding.bge_m3:model:40 - Loading embedding model weights from BAAI/bge-m3...
Loading weights: 100%|██████████| 391/391 [00:01<00:00, 234.21it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [32]:
embeddings = bge_m3.encode(extracted_text.get("title", ""), show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 135.94it/s]


In [33]:
embeddings

array([[-0.02459717,  0.03204346, -0.01960754, ..., -0.00739288,
        -0.02937317,  0.01534271]], shape=(1, 1024), dtype=float32)

In [34]:
embeddings.shape

(1, 1024)